<a href="https://colab.research.google.com/github/AlperYildirim1/geometric-grokking/blob/main/S5_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import json
import itertools
import pandas as pd

# GOOGLE DRIVE SETUP
SAVE_DIR = '/content/drive/MyDrive/Grokking_S5_Negative_Control_1e-3'
os.makedirs(SAVE_DIR, exist_ok=True)

# CONFIGURATION
# We use 1e-3 because task is changed. Previous 6e-4 and 1e-4 configs failed to grok in 100k steps for all models for this task.

N_DEGREE = 5
GROUP_SIZE = math.factorial(N_DEGREE) # 120
FRAC_TRAIN = 0.3
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
LR = 1e-3
MAX_EPOCHS = 100000
LOG_EVERY = 200
SEEDS = list(range(1, 11))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DETERMINISM
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

# S_5 DATASET GENERATION
def generate_s5_elements():
    """Generates the 120 elements of S5 and their composition table."""
    elements = list(itertools.permutations(range(N_DEGREE)))
    elem_to_id = {elem: i for i, elem in enumerate(elements)}
    id_to_elem = {i: elem for i, elem in enumerate(elements)}
    return elements, elem_to_id, id_to_elem

def compose_s5(id1, id2, id_to_elem, elem_to_id):
    """Composes two permutations: applying p2 then p1."""
    p1 = id_to_elem[id1]
    p2 = id_to_elem[id2]
    p_out = tuple(p1[p2[i]] for i in range(N_DEGREE))
    return elem_to_id[p_out]

def make_s5_dataset(frac_train, seed=42):
    set_seed(seed)
    rng = random.Random(seed)

    _, elem_to_id, id_to_elem = generate_s5_elements()

    all_pairs =[(a, b) for a in range(GROUP_SIZE) for b in range(GROUP_SIZE)]
    rng.shuffle(all_pairs)

    n_train = int(len(all_pairs) * frac_train)
    op_token = GROUP_SIZE # 120 acts as the composition operator

    train_x = torch.tensor([[a, b, op_token] for a, b in all_pairs[:n_train]], dtype=torch.long)
    train_y = torch.tensor([compose_s5(a, b, id_to_elem, elem_to_id) for a, b in all_pairs[:n_train]], dtype=torch.long)

    test_x = torch.tensor([[a, b, op_token] for a, b in all_pairs[n_train:]], dtype=torch.long)
    test_y = torch.tensor([compose_s5(a, b, id_to_elem, elem_to_id) for a, b in all_pairs[n_train:]], dtype=torch.long)

    return train_x, train_y, test_x, test_y

# RMSNORM IMPLEMENTATION
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        return x * torch.rsqrt(norm_x + self.eps) * self.weight

# UNIFIED FOR EASE OF USE
class UnifiedTransformer(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model, num_heads, mlp_dim, model_type):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.model_type = model_type

        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(3, d_model)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)

        self.unembed = nn.Linear(d_model, num_classes, bias=False)

        if self.model_type == 'layernorm':
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)
        elif self.model_type == 'rmsnorm':
            self.norm1 = RMSNorm(d_model)
            self.norm2 = RMSNorm(d_model)
            self.norm3 = RMSNorm(d_model)

    def apply_norm(self, x, norm_module=None):
        if self.model_type in ['layernorm', 'rmsnorm']:
            return norm_module(x)
        else: # Both spherical models
            return F.normalize(x, dim=-1)

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x) + self.pos_embed(pos)

        h = self.apply_norm(h, getattr(self, 'norm1', None))

        Q = self.W_Q(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn_out = self.W_O((F.softmax(scores, dim=-1) @ V).transpose(1, 2).contiguous().view(B, L, self.d_model))

        h = h + attn_out
        h = self.apply_norm(h, getattr(self, 'norm2', None))

        h = h + self.mlp_out(F.relu(self.mlp_in(h)))
        h = self.apply_norm(h, getattr(self, 'norm3', None))

        h_final = h[:, 2, :]

        if self.model_type == 'spherical_nowd':
            w_normalized = F.normalize(self.unembed.weight, dim=1)
            logits = F.linear(h_final, w_normalized)
            return logits * 10.0
        else:
            return self.unembed(h_final)

# TRAINING LOGIC
def train_model(model, name, train_x, train_y, test_x, test_y, epochs, weight_decay):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=weight_decay, betas=(0.9, 0.999))
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    grok_epoch = None
    history = {'epochs': [], 'train_acc':[], 'test_acc':[]}
    peak_test_acc = 0.0

    pbar = tqdm(range(epochs), desc=f"Training {name}", leave=False)

    for epoch in pbar:
        model.train()
        logits = model(train_x)
        loss = criterion(logits, train_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                train_acc = (logits.argmax(-1) == train_y).float().mean().item()
                test_logits = model(test_x)
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

                if test_acc > peak_test_acc:
                    peak_test_acc = test_acc

            history['epochs'].append(epoch)
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)

            pbar.set_postfix({'tr_acc': f"{train_acc:.3f}", 'te_acc': f"{test_acc:.3f}"})

            if grok_epoch is None and test_acc >= 0.99:
                grok_epoch = epoch
                tqdm.write(f"\n {name} generalized (>99%) at epoch {epoch}!")

    return {
        "grok_epoch": grok_epoch if grok_epoch else epochs,
        "history": history,
        "peak_acc": peak_test_acc
    }

# EXECUTION & PLOTTING
if __name__ == "__main__":
    print("=" * 60)
    print("S_5 NEGATIVE CONTROL ABLATION (All Models)")
    print("=" * 60)

    summary_results = []

    configs =[
        {"name": "LayerNorm Baseline", "type": "layernorm", "wd": 1.0},
        {"name": "RMSNorm Baseline", "type": "rmsnorm", "wd": 1.0},
        {"name": "Spherical Model (WD=1.0)", "type": "spherical_wd", "wd": 1.0},
        {"name": "Fully Bounded Sphere (WD=0.0)", "type": "spherical_nowd", "wd": 0.0}
    ]

    for SEED in SEEDS:
        print(f"\n\n{'=' * 40}")
        print(f"STARTING RUN FOR SEED {SEED}")
        print(f"{'=' * 40}\n")

        tr_x, tr_y, te_x, te_y = make_s5_dataset(FRAC_TRAIN, seed=SEED)

        # Vocab size is 121 (0-119 elements + 120 op_token), Num classes is 120
        VOCAB_SIZE = GROUP_SIZE + 1
        NUM_CLASSES = GROUP_SIZE

        seed_histories = {}

        for cfg in configs:
            set_seed(SEED)
            model = UnifiedTransformer(
                vocab_size=VOCAB_SIZE,
                num_classes=NUM_CLASSES,
                d_model=D_MODEL,
                num_heads=NUM_HEADS,
                mlp_dim=MLP_DIM,
                model_type=cfg["type"]
            ).to(DEVICE)

            res = train_model(model, cfg["name"], tr_x, tr_y, te_x, te_y, MAX_EPOCHS, weight_decay=cfg["wd"])
            seed_histories[cfg["name"]] = res

            summary_results.append({
                "Architecture": cfg["name"],
                "Seed": SEED,
                "Grok Epoch": res["grok_epoch"],
                "Peak Acc": res["peak_acc"]
            })

            # Save JSON log
            safe_name = cfg["name"].replace(" ", "_").replace("=", "").replace("(", "").replace(")", "")
            save_path = os.path.join(SAVE_DIR, f"S5_{safe_name}_seed{SEED}.json")
            with open(save_path, "w") as f:
                json.dump(res["history"], f)

        # Plot 2x2 Grid for the Seed
        fig, axs = plt.subplots(2, 2, figsize=(14, 10))
        axs = axs.flatten()

        for i, cfg in enumerate(configs):
            name = cfg["name"]
            hist = seed_histories[name]["history"]
            grok_ep = seed_histories[name]["grok_epoch"]

            axs[i].plot(hist["epochs"], hist["train_acc"], label="Train Acc", color="#1f77b4", linewidth=2.5)
            axs[i].plot(hist["epochs"], hist["test_acc"], label="Test Acc", color="#d62728", linewidth=2.5)
            axs[i].fill_between(hist["epochs"], hist["train_acc"], hist["test_acc"], color='grey', alpha=0.15)

            axs[i].set_title(f"{name}\nGrok Epoch: {grok_ep}", fontsize=14, pad=10)
            axs[i].set_xlabel("Epochs", fontsize=12)
            axs[i].set_ylabel("Accuracy", fontsize=12)
            axs[i].set_xlim(0, MAX_EPOCHS)
            axs[i].grid(True, alpha=0.3)
            axs[i].legend(loc="lower right")

        plt.tight_layout()
        plot_path = os.path.join(SAVE_DIR, f"S5_All_Models_seed{SEED}.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

    # Save and Print Summary Table
    df = pd.DataFrame(summary_results)
    df.to_csv(os.path.join(SAVE_DIR, "S5_Summary_Results.csv"), index=False)

    print("\n" + "="*60)
    print("  S5 FINAL SUMMARY RESULTS ")
    print("="*60)
    summary_df = df.groupby("Architecture")["Grok Epoch"].agg(['mean', 'std', 'min', 'max']).reset_index()
    print(summary_df.to_markdown(index=False, tablefmt="github"))

In [ ]:
import time
from google.colab import runtime

time.sleep(30)
runtime.unassign()